# 02 — External 90/10 Split and Drift

**Objective:** Create the production-simulation holdout, prove its isolation, and visualize train/test drift.

In [1]:
from pathlib import Path
import os, sys
import pandas as pd
import plotly.express as px

_cwd = Path.cwd().resolve()
ROOT = _cwd.parent if _cwd.name == "dev" else _cwd
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("MPLCONFIGDIR", "/tmp/credit-risk-lab-matplotlib")

from credit_risk_lab.config.settings import settings
print(f"Project root: {settings.project_root}")

Project root: /Users/surelmanda/Mlops-Databricks-Projects/credit-risk-lab


## 1. Create and persist the external split

`test.csv` remains raw so API simulation exercises feature engineering at inference time.

In [2]:
from credit_risk_lab.application import create_deployment_split
from credit_risk_lab.infrastructure.data_sources import CSVDataSourceConfig, CSVDatasetRepository

raw_source = CSVDataSourceConfig(path=settings.raw_data_path, sep=settings.raw_data_sep, encoding=settings.raw_data_encoding)
raw_df = CSVDatasetRepository(raw_source).load()
split_result = create_deployment_split(raw_df, test_size=0.10)
split_result

2026-07-11 09:51:58 | INFO     | CSVDatasetRepository | credit_risk_lab.infrastructure.data_sources.csv_dataset_repository:load:51 - Chargement du fichier : /Users/surelmanda/Mlops-Databricks-Projects/credit-risk-lab/data/raw/loan_data.csv


2026-07-11 09:51:58 | INFO     | CSVDatasetRepository | credit_risk_lab.infrastructure.data_sources.csv_dataset_repository:load:59 - Dataset chargé (45000 lignes, 14 colonnes)


DeploymentSplitResult(train_path=PosixPath('/Users/surelmanda/Mlops-Databricks-Projects/credit-risk-lab/data/processed/train.csv'), test_path=PosixPath('/Users/surelmanda/Mlops-Databricks-Projects/credit-risk-lab/data/processed/test.csv'), train_rows=40493, test_rows=4500)

## 2. Load both persisted partitions explicitly

In [3]:
train_df = CSVDatasetRepository(CSVDataSourceConfig(path=settings.train_path)).load()
test_df = CSVDatasetRepository(CSVDataSourceConfig(path=settings.test_path)).load()
pd.DataFrame({
    "sample": ["train", "external_test"],
    "rows": [len(train_df), len(test_df)],
    "positive_rate": [train_df[settings.target_column].mean(), test_df[settings.target_column].mean()],
})

2026-07-11 09:51:58 | INFO     | CSVDatasetRepository | credit_risk_lab.infrastructure.data_sources.csv_dataset_repository:load:51 - Chargement du fichier : /Users/surelmanda/Mlops-Databricks-Projects/credit-risk-lab/data/processed/train.csv


2026-07-11 09:51:58 | INFO     | CSVDatasetRepository | credit_risk_lab.infrastructure.data_sources.csv_dataset_repository:load:59 - Dataset chargé (40493 lignes, 14 colonnes)


2026-07-11 09:51:58 | INFO     | CSVDatasetRepository | credit_risk_lab.infrastructure.data_sources.csv_dataset_repository:load:51 - Chargement du fichier : /Users/surelmanda/Mlops-Databricks-Projects/credit-risk-lab/data/processed/test.csv


2026-07-11 09:51:58 | INFO     | CSVDatasetRepository | credit_risk_lab.infrastructure.data_sources.csv_dataset_repository:load:59 - Dataset chargé (4500 lignes, 14 colonnes)


,sample,rows,positive_rate
0,train,40493,0.222261
1,external_test,4500,0.222222


## 3. Complete drift report

Numerical features receive PSI, KS, and Hellinger; categorical features receive PSI over the category union.

In [4]:
from credit_risk_lab.infrastructure.analytics import DriftAnalyzer

features = [c for c in train_df.columns if c != settings.target_column]
drift_report = DriftAnalyzer(bins=10).report_frame(train_df, test_df, features=features)
drift_report

,feature,type,psi,ks,hellinger,status
0,cb_person_cred_hist_length,numeric,0.004132,0.022120,0.022724,stable
1,person_income,numeric,0.003801,0.015228,0.021795,stable
2,person_emp_exp,numeric,0.003661,0.023364,0.021391,stable
3,person_age,numeric,0.002906,0.019347,0.019057,stable
4,credit_score,numeric,0.002711,0.018315,0.018406,stable
5,loan_amnt,numeric,0.002138,0.012297,0.016349,stable
6,loan_int_rate,numeric,0.001950,0.012991,0.015613,stable
7,loan_intent,categorical,0.001535,NaN,NaN,stable
8,loan_percent_income,numeric,0.001134,0.008615,0.011906,stable
9,person_home_ownership,categorical,0.000436,NaN,NaN,stable


## 4. Global PSI visual and detailed distributions

In [5]:
from credit_risk_lab.infrastructure.visualization import (
    plot_categorical_distribution, plot_drift_summary, plot_numeric_distribution,
)

plot_drift_summary(drift_report).show()
for feature in drift_report.query("type == 'numeric'")["feature"].head(3):
    plot_numeric_distribution(train_df, test_df, feature).show()
for feature in drift_report.query("type == 'categorical'")["feature"].head(2):
    plot_categorical_distribution(train_df, test_df, feature).show()

## 5. Persist drift artifacts

In [6]:
settings.reports_dir.mkdir(parents=True, exist_ok=True)
drift_report.to_csv(settings.drift_report_path, index=False)
plot_drift_summary(drift_report).write_html(settings.drift_summary_plot_path, include_plotlyjs="cdn")
{"csv": str(settings.drift_report_path), "html": str(settings.drift_summary_plot_path)}

{'csv': '/Users/surelmanda/Mlops-Databricks-Projects/credit-risk-lab/reports/deployment_split_drift.csv',
 'html': '/Users/surelmanda/Mlops-Databricks-Projects/credit-risk-lab/reports/deployment_split_drift.html'}